# BS49903 · DS/AI for Biology
## Week 2 Practice Notebook — Data Structures & Basic Programming

Covers Week 2-1 (Data Structures) and Week 2-2 (Basic Programming).

In [3]:
import numpy as np

---
# Part 1 · Data types and numbers

### 1.1 The three basic scalar types

In [24]:
i = 3            # int
f = 3.14         # float
b = True         # bool

print(i, type(i))
print(f, type(f))
print(b, type(b))

3 <class 'int'>
3.14 <class 'float'>
True <class 'bool'>


### 1.2 Integer and float arithmetic

Note which operations return an `int` and which return a `float`.

In [25]:
print(1 + 3)        # int
print(9 - 5)        # int
print(5 * 4)        # int
print(8 / 3)        # float  <- division ALWAYS gives a float
print(8 // 3)       # int    <- floor division, drops the remainder
print(8 % 3)        # int    <- modulo: the remainder
print(2 ** 4)       # int    <- power
print(1 + 2.5)      # float  <- int + float promotes to float

4
4
20
2.6666666666666665
2
2
16
3.5


`%` (modulo) is the one to remember — it is how you test divisibility:

`n % 2 == 0` means *n is even*.

In [26]:
for n in [4, 7, 10, 15]:
    print(n, "even?" , n % 2 == 0)

4 even? True
7 even? False
10 even? True
15 even? False


### 1.3 Fixed-width integer types and overflow

Plain Python `int` grows as large as it needs to. NumPy integers do **not** — they have a
fixed number of bits, and a `uint8` can only hold 0–255.

In [27]:
print(2 ** 100)     # plain Python int: no limit

1267650600228229401496703205376


`a = 242` and `b = 100`, both stored as `uint8`. What is `a + b`?

In [28]:
a = np.array(242, dtype=np.uint8)
b = np.array(100, dtype=np.uint8)

print(a + b)        # not 342!

86


242 + 100 = 342, but 342 does not fit in 8 bits. The value wraps around:
342 - 256 = **86**. NumPy warns you, but it does not stop you — silent overflow is a
real source of bugs in image and sequencing data.

In [29]:
print("uint8  range:", np.iinfo(np.uint8).min,  "to", np.iinfo(np.uint8).max)
print("int8   range:", np.iinfo(np.int8).min,   "to", np.iinfo(np.int8).max)
print("uint16 range:", np.iinfo(np.uint16).min, "to", np.iinfo(np.uint16).max)

uint8  range: 0 to 255
int8   range: -128 to 127
uint16 range: 0 to 65535


### 1.4 Why the data type matters: memory

An image with pixel values 0–255, size 9800 × 9792:

In [30]:
h, w = 9800, 9792
pixels = h * w

for dtype in [np.uint64, np.uint32, np.uint8]:
    nbytes = pixels * np.dtype(dtype).itemsize
    print(f"{np.dtype(dtype).name:>7}: {nbytes / 1e6:7.1f} MB")

 uint64:   767.7 MB
 uint32:   383.8 MB
  uint8:    96.0 MB


Same image, same information — 8× the memory if you pick the wrong type.

---
# Part 2 · Booleans and comparison

### 2.1 Comparison operators

In [31]:
print(3 == 3)            # equal to
print("abc" != "ab")     # not equal to
print(3 > 5, 3 < 5)
print(3 >= 3, 3 <= 2)
print([1, 2] == [1, 2])  # lists compare element by element, in order
print([2, 1] == [1, 2])  # order matters

True
True
False True
True False
True
False


### 2.2 Booleans are numbers

What is `True + True`?

In [32]:
print(True * True)
print(True - True)
print(True + True)       # True behaves as 1, False as 0
print(sum([True, False, True, True]))   # a handy way to count matches

1
0
2
3


### 2.3 Truthiness

Any value can be used where a condition is expected. `0`, `0.0`, `""`, `[]`, `{}`
and `None` are **falsy**; everything else is **truthy**.

In [33]:
for value in [0, 1, -5, 0.0, "", "ATG", [], [0], {}, None]:
    print(repr(value), "->", bool(value))

0 -> False
1 -> True
-5 -> True
0.0 -> False
'' -> False
'ATG' -> True
[] -> False
[0] -> True
{} -> False
None -> False


---
# Part 3 · Logical vs. bitwise operators

### 3.1 `and`, `or`, `not`

In [34]:
print(True and True)
print(True or False)
print(not True)

True
True
False


Important detail: `and` and `or` return **one of the operands**, not `True`/`False`.

- `a or b` → returns `a` if `a` is truthy, otherwise `b`
- `a and b` → returns `a` if `a` is falsy, otherwise `b`

In [35]:
print(2 or 1)        # 2 is truthy, so it is returned; 1 is never looked at
print(0 or 1)        # 0 is falsy, so the second operand is returned
print(2 and 1)
print(0 and 1)
print(bool(2 or 1))  # wrap in bool() if you really want True/False

2
1
1
0
True


This is also why the *order of operands* matters — `and`/`or` **short-circuit**,
stopping as soon as the answer is known.

In [36]:
x = None

print(x is not None and x > 0)   # safe: the second test never runs

try:
    print(x > 0 and x is not None)   # unsafe: crashes on the first test
except TypeError as e:
    print("TypeError:", e)

False
TypeError: '>' not supported between instances of 'NoneType' and 'int'


### 3.2 `&`, `|`, `~` work on bits, not on truth

`&` compares two integers **one bit at a time**.

```
  12 = 1100
  10 = 1010
  ---------
   &   1000  = 8
   |   1110  = 14
   ^   0110  = 6
```

In [37]:
print(12 & 10, bin(12), bin(10), bin(12 & 10))
print(12 | 10, bin(12 | 10))
print(12 ^ 10, bin(12 ^ 10))    # XOR: exactly one of the two bits is 1
print(3 & 2, 3 | 2)

8 0b1100 0b1010 0b1000
14 0b1110
6 0b110
2 3


In [38]:
print(5 << 2)      # shift bits left  -> multiply by 2**2
print(20 >> 2)     # shift bits right -> divide by 2**2

20
5


**A caution about `~` on plain integers.** Python integers are signed, so flipping every
bit of `12` does not give `3`. It gives `-13`: the rule is `~n == -n - 1`.
On boolean arrays, though, `~` behaves exactly like "NOT" — which is how you will use it.

In [39]:
print(~12)                  # -13, not 3
print(~np.array(True))      # on booleans it is a plain NOT

-13
False


### 3.3 `&` and `|` on NumPy boolean arrays — this is where they earn their keep

In [40]:
a = np.array([True, False, True])
b = np.array([True, False, False])

print(a & b)     # element-wise AND
print(a | b)     # element-wise OR
print(~a)        # element-wise NOT

[ True False False]
[ True False  True]
[False  True False]


### 3.4 Boolean masking

This is the single most useful pattern in the whole lecture: use a boolean array to
**select** elements. It replaces a `for` loop with an `if` inside it.

In [41]:
expression = np.array([2.1, 0.3, 5.8, 0.1, 3.4])

mask = expression > 1
print("mask:      ", mask)
print("selected:  ", expression[mask])
print("in one line:", expression[expression > 1])
print("how many:  ", (expression > 1).sum())    # True counts as 1

mask:       [ True False  True False  True]
selected:   [2.1 5.8 3.4]
in one line: [2.1 5.8 3.4]
how many:   3


In [42]:
# Combining conditions: the parentheses are REQUIRED,
# because & binds more tightly than >
print(expression[(expression > 1) & (expression < 5)])
print(expression[~(expression > 1)])            # the ones we excluded

[2.1 3.4]
[0.3 0.1]


What happens without the parentheses? Run it and read the error.

In [43]:
try:
    expression[expression > 1 & expression < 5]
except Exception as e:
    print(type(e).__name__, ":", e)

TypeError : ufunc 'bitwise_and' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''


---
# Part 4 · Lists, tuples, strings, dictionaries, arrays

### 4.1 List — ordered, mixed types allowed, **mutable**

In [44]:
a = [1, 2]
b = [3, 4, 5]
c = [1, 3, 2, 2, 2]

print(a[0])        # indexing starts at 0
print(b[1])
print(c[1:4])      # slicing: start included, stop excluded
print(c[-1])       # negative index counts from the end
print(len(c))

1
4
[3, 2, 2]
2
5


In [45]:
c = [1, 3, 2, 2, 2]
c[2] = 5           # lists are mutable: you can assign to an element
print(c)

[1, 3, 5, 2, 2]


In [46]:
a = [1, 2]
b = [3, 4, 5]

print(a + b)       # concatenation
print(a * 3)       # repetition
print(b * 2)

[1, 2, 3, 4, 5]
[1, 2, 1, 2, 1, 2]
[3, 4, 5, 3, 4, 5]


Lists can hold anything, including other lists:

In [47]:
mixed  = [1, 2, "apple", "pear", [4, 5]]
nested = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
print(mixed)
print(nested[2][1])     # element 1 of list 2

[1, 2, 'apple', 'pear', [4, 5]]
7


### 4.2 List methods

Notice that `append`, `remove`, `sort` and `reverse` return `None` — they change the
list **in place**. `pop` is the exception: it returns the element it removed.

In [48]:
a = [1, 2]
print(a.append(9))     # returns None ...
print(a)               # ... but a has changed

None
[1, 2, 9]


In [49]:
b = [3, 4, 5]
print(b.pop())         # pop returns the removed element
print(b)

5
[3, 4]


In [50]:
c = [1, 3, 2, 2, 2]
c.remove(2)            # removes the FIRST matching element only
print(c)
print(c.count(2))      # how many 2s are left

[1, 3, 2, 2]
2


In [51]:
b = [3, 4, 5]
c = [1, 3, 2, 2, 2]
b.reverse(); print(b)
c.sort();    print(c)

[5, 4, 3]
[1, 2, 2, 2, 3]


### 4.3 Tuple — like a list, but **immutable**

In [52]:
a = (1, 2)
b = (3, 4, 5)
c = (1, 3, 2, 2, 2)

print(a[0], b[1], c[1:4])
print(a + b)
print(a * 3)

1 4 (3, 2, 2)
(1, 2, 3, 4, 5)
(1, 2, 1, 2, 1, 2)


In [53]:
c = (1, 3, 2, 2, 2)
try:
    c[2] = 5
except TypeError as e:
    print("TypeError:", e)

TypeError: 'tuple' object does not support item assignment


### 4.4 String — a sequence of characters, also immutable

In [54]:
a = "DataScience"
b = "and"
c = "AI"
d = "  bio, math, cs  "

print(a[0])
print(a[0:4])
print(a + b + c)
print(c * 3)
print(len(a))

D
Data
DataScienceandAI
AIAIAI
11


In [55]:
a = "DataScience"
try:
    a[1] = "e"
except TypeError as e:
    print("TypeError:", e)

TypeError: 'str' object does not support item assignment


In [56]:
print(c.lower())
print(a.upper())
print(repr(d.strip()))     # repr() shows the quotes, so you can see the spaces
print(repr(d.lstrip()))
print(repr(d.rstrip()))

ai
DATASCIENCE
'bio, math, cs'
'bio, math, cs  '
'  bio, math, cs'


In [57]:
print(d.split(","))
print(",".join(["bio", "math", "cs"]))
print("/".join(["bio", "math", "cs"]))
print("ATGCGC".count("G"))
print("ATG" in "ATGCGC")       # membership test

['  bio', ' math', ' cs  ']
bio,math,cs
bio/math/cs
2
True


### 4.5 Dictionary — key → value lookup

In [58]:
card = {"face": "spade", "value": 10}
print(card["face"])
print(card["value"])

spade
10


In [59]:
letter = {"a": 1, "b": 2, "c": 3, "d": 4}
print(letter.keys())
print(letter.values())

letter["e"] = 5            # add a new key
print(letter)
print("z" in letter)       # membership tests the KEYS
print(letter.get("z", 0))  # safe lookup with a default

dict_keys(['a', 'b', 'c', 'd'])
dict_values([1, 2, 3, 4])
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
False
0


A dictionary is the natural way to store a lookup table — a codon table, for example:

In [60]:
codon_table = {"AUG": "Met", "UUU": "Phe", "UUC": "Phe",
               "UAA": "Stop", "UAG": "Stop", "UGA": "Stop"}
print(codon_table["UGA"])

Stop


### 4.6 NumPy array — fixed size, fixed dtype, optimized for math

In [62]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[1, 2], [1, 1]])
c = np.array([[1, 2], [3, 4], [5, 6]])

print(a)
print("a.shape:", a.shape)
print("c.shape:", c.shape)
print("a.dtype:", a.dtype)

[[1 2]
 [3 4]]
a.shape: (2, 2)
c.shape: (3, 2)
a.dtype: int64


In [63]:
a8 = np.array([[1, 2], [3, 4]], dtype="uint8")
print(a8.dtype)
print(a8.astype("float32").dtype)

uint8
float32


In [64]:
print(a + b)      # element-wise
print(a * b)      # element-wise, NOT matrix multiplication

[[2 4]
 [4 5]]
[[1 4]
 [3 4]]


In [65]:
try:
    a + c         # shapes (2,2) and (3,2) are incompatible
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (2,2) (3,2) 


In [66]:
print(a.dot(b))          # matrix multiplication
print(np.matmul(a, b))   # same thing
print(a @ b)             # and the same again, shorter
print(c.T)               # transpose

[[ 3  4]
 [ 7 10]]
[[ 3  4]
 [ 7 10]]
[[ 3  4]
 [ 7 10]]
[[1 3 5]
 [2 4 6]]


### 4.7 Type casting

In [67]:
a = 123
b = "123"
print(type(a))

a = str(a);   print(repr(a), type(a))
b = int(b);   print(b, type(b))
a = float(a); print(a, type(a))

<class 'int'>
'123' <class 'str'>
123 <class 'int'>
123.0 <class 'float'>


In [68]:
lst = [1, 2, 3]
arr = np.array([1, 2, 3])

print(np.array(lst))
print(list(arr))
print(arr.astype("uint8"))

[1 2 3]
[np.int64(1), np.int64(2), np.int64(3)]
[1 2 3]


What does `int('3.7')` do? And `int(3.7)`?

In [69]:
print(int(3.7))          # truncates toward zero
try:
    int("3.7")
except ValueError as e:
    print("ValueError:", e)

3
ValueError: invalid literal for int() with base 10: '3.7'


---
# Part 5 · Conditional statements

### 5.1 `if` / `else`

In [70]:
score = 90

if score >= 60:
    print("Pass")
else:
    print("Fail")

Pass


The condition does not have to be a comparison — any truthy value works:

In [71]:
score = 90

if score:            # 90 is truthy
    print("Pass")
else:
    print("Fail")

score = 0            # careful: a score of 0 is FALSY
if score:
    print("Pass")
else:
    print("Fail")

Pass
Fail


In [72]:
age = 22
has_id = True

if (age >= 21) and has_id:
    print("Entry permitted")
else:
    print("Entry denied")

Entry permitted


### 5.2 Order matters: the first match wins

Only the **first** matching branch runs. The rest are skipped, even if they also match.

In [73]:
score = 90

if score >= 90:
    print("A")
elif score >= 80:
    print("B")
elif score >= 70:
    print("C")
else:
    print("D")

A


The same score, the same thresholds — but reordered. What prints?

In [74]:
score = 90

if score >= 70:
    print("C")
elif score >= 80:
    print("B")
elif score >= 90:
    print("A")
else:
    print("D")

C


`score >= 70` is true, so it wins and nothing below it is even tested. The `elif`
branches for B and A are unreachable for any score above 70.

**Fix 1:** order from most specific to least specific (the first version).
**Fix 2:** make the conditions non-overlapping, so order cannot matter.

In [75]:
score = 90

if   (score >= 70) and (score < 80):
    print("C")
elif (score >= 80) and (score < 90):
    print("B")
elif (score >= 90) and (score <= 100):
    print("A")
else:
    print("D")

A


### 5.3 `match` / `case`

In [76]:
codon = "UGA"

match codon:
    case "AUG":
        print("Methionine (Start)")
    case "UUU" | "UUC":
        print("Phenylalanine")
    case "UUA" | "UUG":
        print("Leucine")
    case "UAA" | "UAG" | "UGA":
        print("Stop codon")
    case _:
        print("Unknown codon")

Stop codon


For a pure lookup like this, a dictionary is usually shorter and easier to extend:

In [77]:
codon_table = {"AUG": "Methionine (Start)",
               "UUU": "Phenylalanine", "UUC": "Phenylalanine",
               "UUA": "Leucine",       "UUG": "Leucine",
               "UAA": "Stop codon",    "UAG": "Stop codon", "UGA": "Stop codon"}

print(codon_table.get("UGA", "Unknown codon"))
print(codon_table.get("XYZ", "Unknown codon"))

Stop codon
Unknown codon


---
# Part 6 · Loops

### 6.1 `for` with `range()` — repeat a fixed number of times

In [78]:
for i in range(5):
    print(i)

0
1
2
3
4


In [79]:
print(list(range(5)))         # 0 to 4  — the stop value is EXCLUDED
print(list(range(2, 7)))      # 2 to 6
print(list(range(2, 10, 2)))  # start, stop, step

print(np.arange(5))
print(np.arange(2, 7))
print(np.arange(2, 10, 2))

[0, 1, 2, 3, 4]
[2, 3, 4, 5, 6]
[2, 4, 6, 8]
[0 1 2 3 4]
[2 3 4 5 6]
[2 4 6 8]


### 6.2 Looping over the items directly

If you do not need the index, do not create one. This is the more Pythonic loop, and it
works on lists, tuples and strings alike.

In [80]:
genes = ["TP53", "BRCA1", "EGFR"]

for gene in genes:
    print(gene)

TP53
BRCA1
EGFR


In [81]:
for base in "ATGC":
    print(base)

A
T
G
C


If you *do* need the position as well, use `enumerate`:

In [82]:
for i, base in enumerate("ATGC"):
    print(i, base)

0 A
1 T
2 G
3 C


### 6.3 `for` + `if`: act only on the iterations you care about

In [84]:
for i in range(1000):
    if i == 100:
        print(i)
    elif i == 500:
        print(i)

100
500


### 6.4 The accumulator pattern

Initialize a variable **before** the loop, update it **inside**, use it **after**.
This is the most transferable idea in the lecture — counting, summing, finding a maximum
and building a list are all the same shape.

In [85]:
seq = "ATGCGCTAAGC"

gc = 0                      # 1. initialize before
for base in seq:            # 2. update inside
    if base in "GC":
        gc += 1             #    += is shorthand for gc = gc + 1

print("GC count:  ", gc)    # 3. use after
print("GC content:", gc / len(seq) * 100, "%")

GC count:   6
GC content: 54.54545454545454 %


### 6.5 `while` — repeat until a condition stops being true

Note the lowercase `while`. And note that *something inside the loop must eventually make
the condition false*, or the loop never ends.

In [86]:
i = 0
while i < 5:
    print(i)
    i = i + 1

0
1
2
3
4


The cell below is an **infinite loop**. It is commented out on purpose — `i` is never
updated, so `i < 5` stays true forever. If you run one by accident, press the ⏹ stop
button (or Kernel → Interrupt).

```python
i = 0
while i < 5:
    print(i)      # no i = i + 1, so this never stops
```

### 6.6 `break` — leave the loop immediately

In [87]:
for i in range(10):
    print(i)
    if i == 3:
        break        # exits the nearest enclosing loop

0
1
2
3


In [88]:
i = 0
while True:          # deliberately infinite ...
    print(i)
    if i == 3:
        break        # ... with break as the only way out
    i = i + 1

0
1
2
3


A more realistic use — stop searching as soon as you find what you want:

In [89]:
seq = "GCGATGCGCTAA"

for i in range(len(seq) - 2):
    codon = seq[i:i+3]
    if codon == "ATG":
        print("Start codon found at position", i)
        break            # no reason to keep looking

Start codon found at position 3


### 6.7 `continue` — skip the rest of this iteration

In [90]:
for i in range(6):
    if i % 2 == 0:
        continue      # even -> jump straight to the next iteration
    print(i)

1
3
5


The usual purpose: skip data you do not want to process.

In [91]:
l = [[1, 1, 2], [1, 2, 3], [], [], [2, 4, 6]]

for data in l:
    if data == []:
        continue
    print(data.count(1))

2
1
0


### 6.8 Nested loops

In [92]:
l = [[1, 1, 2], [1, 2, 3], [], [], [2, 4, 6]]

for data in l:          # outer loop: one list at a time
    c = 0
    for i in data:      # inner loop: one element at a time
        if i == 1:
            c = c + 1
    print(c)

2
1
0
0
0


Same idea, but now `break` appears in the inner loop.

In [93]:
l = [[1, 1, 2, 1, 1], [1, 2, 1, 3], [2, 1, 1, 1]]

for data in l:
    c = 0
    for i in data:
        if i == 1:
            c = c + 1
        elif i == 2:
            break       # breaks the INNER loop only
    print(c)

2
1
0


`break` exits only the loop it sits in, so the outer loop carries on with the next
sublist. Counting stops at the first `2` in each row: 2, 1, 0.

---
# Part 7 · Functions

### 7.1 Definition and call

```
def  func(x1, x2):      <- name, and parameters to process
     y = ...            <- the work
     return y           <- the output
```

In [94]:
def double(x):
    y = 2 * x
    return y

print(double(5))
print(double(1.5))
print(double("ab"))     # works on anything that supports * — watch out

10
3.0
abab


In [95]:
def double(x):
    return 2 * x        # shorter, same behaviour

print(double(5))

10


### 7.2 A function with no `return`

It still returns something: `None`. This is the difference between *printing* a value and
*returning* one.

In [96]:
def greet(name):
    print(f"Hello, {name}!")

greet("Ana")

result = greet("Ana")
print("result is:", result)      # None

Hello, Ana!
Hello, Ana!
result is: None


In [97]:
def add_score(scores, value):
    scores.append(value)         # useful work, no return value

my_scores = []
add_score(my_scores, 88)
add_score(my_scores, 95)
print(my_scores)

[88, 95]


### 7.3 Local and global variables

In [99]:
def double(x):
    a = 2                # a is LOCAL: it exists only inside the function
    y = a * x
    return y

print(double(5))

try:
    print(a)             # not visible out here
except NameError as e:
    print("NameError:", e)

10
123.0


In [100]:
a = 2                    # a is GLOBAL

def double(x):
    y = a * x            # reading a global is fine
    return y

print(double(5))

10


In [101]:
a = 2

def multiply(x):
    a = 3                # this creates a NEW local a; the global one is untouched
    y = a * x
    return y

print(multiply(5))
print("global a is still:", a)

15
global a is still: 2


The global `a` is 2, and the function starts with `a = a + 1`. What happens?

In [102]:
a = 2

def multiply(x):
    a = a + 1            # assigning to a makes a local for the WHOLE function,
    y = a * x            # so the a on the right-hand side does not exist yet
    return y

try:
    print(multiply(5))
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

UnboundLocalError: cannot access local variable 'a' where it is not associated with a value


### 7.4 Putting it together

In [104]:
def gc_content(seq):
    """Return the GC content of a DNA sequence, as a percentage."""
    gc = 0
    for base in seq:
        if base in "GC":
            gc += 1
    return gc / len(seq) * 100

for s in ["ATGCGCTAAGC", "AAATTT", "GCGCGC"]:
    print(s, "->", round(gc_content(s), 2), "%")

ATGCGCTAAGC -> 54.55 %
AAATTT -> 0.0 %
GCGCGC -> 100.0 %


---
# Part 8 · Classes and objects

A **class** is a definition — a blueprint. An **object** is a thing made from it.
Attributes are what an object *has*; methods are what it *does*.

In [105]:
class DNAseq:

    def __init__(self, seq):        # runs automatically when you create an object
        self.seq = seq              # attribute

    def length(self):               # method
        return len(self.seq)

    def gc_content(self):           # method
        gc = 0
        for base in self.seq:
            if base in "GC":
                gc += 1
        return gc / len(self.seq) * 100

In [106]:
gene = DNAseq("ATGCGCTAAGC")

print(gene.seq)             # attribute: no parentheses
print(gene.length())        # method: parentheses
print(gene.gc_content())

ATGCGCTAAGC
11
54.54545454545454


### 8.1 `self` — one class, many objects

`self` is the object the method was called on. That is how the same method gives
different answers for different objects.

In [107]:
gene1 = DNAseq("ATGCGCTAAGC")
gene2 = DNAseq("AAATTTAAA")

print(gene1.gc_content())
print(gene2.gc_content())   # same method, different self

54.54545454545454
0.0


### 8.2 Magic (dunder) methods

`print(gene)` is not very informative right now. Special methods hook your class into
syntax that already exists: `print()` calls `__str__`, `len()` calls `__len__`.

In [110]:
print(gene)      # <__main__.DNAseq object at 0x...>  — not helpful

try:
    len(gene)
except TypeError as e:
    print("TypeError:", e)

TypeError: object of type 'DNAseq' has no len()


In [111]:
class DNAseq:

    def __init__(self, seq):
        self.seq = seq

    def __str__(self):                  # print(obj)
        return f"DNAseq({self.seq})"

    def __len__(self):                  # len(obj)
        return len(self.seq)

    def __getitem__(self, i):           # obj[i], and iteration for free
        return self.seq[i]

    def gc_content(self):
        gc = 0
        for base in self.seq:
            if base in "GC":
                gc += 1
        return gc / len(self.seq) * 100

    def reverse_complement(self):
        pairs = {"A": "T", "T": "A", "G": "C", "C": "G"}
        out = ""
        for base in reversed(self.seq):
            out = out + pairs[base]
        return out

In [112]:
gene = DNAseq("ATGCGCTAAGC")

print(gene)                       # __str__
print(len(gene))                  # __len__
print(gene[0], gene[0:3])         # __getitem__
print(gene.gc_content())
print(gene.reverse_complement())

DNAseq(ATGCGCTAAGC)
11
A ATG
54.54545454545454
GCTTAGCGCAT


In [114]:
for base in gene:                 # iteration comes free with __getitem__
    print(base, end=" ")

A T G C G C T A A G C 

### 8.3 Everything you have already used is a class

`list`, `str`, `tuple`, `dict` and `np.ndarray` are all classes. That is why they have
attributes and methods reached with a dot — exactly like your `DNAseq`.

In [5]:
a = [1, 2, 3, 4, 5]
print(type(a))
print(a.count(1), a.index(3))
a.append(6); print(a)

<class 'list'>
1 2
[1, 2, 3, 4, 5, 6]


In [4]:
a = np.array([1, 2, 3, 4, 5], dtype="uint8")
print(type(a))
print("attributes:", a.shape, a.dtype, a.ndim, a.size)
print("methods:   ", a.min(), a.max(), a.mean(), a.sum())
print(a.astype("float32").dtype)

<class 'numpy.ndarray'>
attributes: (5,) uint8 1 5
methods:    1 5 3.0 15
float32
